# Post 2: a 20-block GPT on web text, on a Colab accelerator

Trains the model in `train.RUNS["d20"]` on ClimbMix, the web text nanochat uses, with the
tokenizer from `tokenizer/bpe.py`. Text is tokenized once, ahead of training.

Checkpoints and token files go straight to a GCS bucket and metrics go to Weights &
Biases, so all of them outlive the runtime. **If the runtime goes away, start a new one
and run the notebook again: it reuses the token files and carries on from the newest
checkpoint.**

`COLAB.md` explains how to run this with `colab exec`, which runs every cell in order with
nobody at the keyboard. With `--env SMOKE=1`, the same notebook runs `train.RUNS["smoke"]`
on two shards, which proves every step in minutes on any runtime.

## Setup

On Colab this clones the branch below and installs it. `BRANCH` must already be pushed,
because the runtime sees GitHub, not your working copy. Locally it does nothing.

The repo needs a JAX recent enough to have `jax.set_mesh`. If the runtime's is older,
this upgrades it before anything imports JAX.

In [ ]:
import importlib.util
import os
import pathlib
import shutil
import sys

BRANCH = "p02-tinystories"
SMOKE = os.environ.get("SMOKE") == "1"

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/toylib").exists():
        !git clone --depth 1 --branch {BRANCH} https://github.com/anujkhare/toylib.git /content/toylib
    %cd /content/toylib
    !git pull --ff-only origin {BRANCH}
    !pip install -q -e ".[harness,viz,gcs,webtext]"
    has_set_mesh = !python -c "import jax; print(hasattr(jax, 'set_mesh'))"
    if has_set_mesh[-1] != "True":
        extra = "[cuda12]" if shutil.which("nvidia-smi") else "[tpu]" if importlib.util.find_spec("libtpu") else ""
        !pip install -q -U "jax{extra}"

## Sign in to GCS and Weights & Biases

`colab exec` can't answer prompts, so this cell asks for nothing. `COLAB.md` shows how a
service-account key and the W&B key reach the runtime as files.

If the runtime can't read the bucket, this stops here rather than failing hours later.

In [ ]:
WANDB_PROJECT = "toylib-p02-climbmix"
WANDB_ENTITY = "anujkhare18"
BUCKET = "gs://tinystories-checkpoints"
WANDB_KEY_FILE = pathlib.Path("/content/.wandb_key")

if IN_COLAB:
    project = os.environ.get("GOOGLE_CLOUD_PROJECT")
    if project:
        !gcloud config set project {project}
    bucket_access = !gcloud storage ls {BUCKET}/ 2>&1 | head -1
    if not bucket_access or not bucket_access[0].startswith("gs://"):
        raise RuntimeError(f"Can't read the bucket, so check the service-account key; see COLAB.md: {bucket_access}")
    if "WANDB_API_KEY" not in os.environ:
        if not WANDB_KEY_FILE.exists():
            raise RuntimeError(f"Upload the W&B key to {WANDB_KEY_FILE} first; see COLAB.md.")
        os.environ["WANDB_API_KEY"] = WANDB_KEY_FILE.read_text().strip()

## The run

| Setting | Value |
| --- | --- |
| Data | ClimbMix, 120 shards, about 6.4B tokens |
| Blocks / dim / heads | 20 / 1,280 / 10 |
| Vocabulary | 32,768, trained on 8 shards |
| Context | 2,048 |
| Parameters | 477M |
| Batch | 64, as 4 microbatches of 16 |
| Tokens seen | 5.7B, 12 per parameter |
| Precision | bfloat16 for compute, float32 for parameters |

Learning rates are the ones that trained well on small models: Muon at 0.02 on the blocks,
and Adam at 0.2 and 0.004 on the embeddings and output layer, scaled by the square root of
768 / 1,280, with warmup and cosine decay. nanochat's defaults have since moved to 0.3 and
0.008, so those are the next thing to try.

Gradient checkpointing (remat) trades speed for memory, and is on. If this runs out of
memory, halve the batch size and double the microbatches in `train.RUNS["d20"]`, then push.

The run keeps the newest three checkpoints, about 6 GB each.

In [ ]:
import jax
from absl import logging as absl_logging

from toylib.harness import analyze
from toylib_projects.p02_gpt_web import experiment, train, tokenize_webtext

RUN_ID = "smoke" if SMOKE else "d20"
RUN = train.RUNS[RUN_ID]
TIMING_STEPS = 3 if SMOKE else 20

ROOT = pathlib.Path("/content" if IN_COLAB else train.OUTPUT_DIR)
DATA_DIR = pathlib.Path(os.environ.get("DATA_DIR", ROOT / "climbmix"))
CHECKPOINT_ROOT = f"{BUCKET}/toylib-p02-climbmix" if IN_COLAB else str(ROOT / "climbmix_runs")
DATA_BUCKET = f"{BUCKET}/toylib-p02-climbmix/data_shards{RUN.num_train_shards}"

absl_logging.set_verbosity(absl_logging.ERROR)  # quiet orbax's read-only notices
print("JAX", jax.__version__, "| devices:", jax.devices())
for device in jax.local_devices():
    limit = (device.memory_stats() or {}).get("bytes_limit")
    print(f"{device.device_kind}: {limit / 2**30:.1f} GiB" if limit else device.device_kind)

## Tokens

The tokenizer is already in the bucket. The first run downloads the shards, writes the
token files and copies them to the bucket, and later runs only copy them back. Measured on
a laptop, a shard takes about 35 seconds on one core.

This needs no accelerator. To avoid paying for one during this step, run the notebook once
on a CPU runtime: it stops after this cell, with the token files in the bucket.

In [ ]:
token_files = [
    tokenize_webtext.tokenizer_path(RUN.vocab_size, DATA_DIR),
    tokenize_webtext.bin_path("train", RUN.vocab_size, DATA_DIR),
    tokenize_webtext.bin_path("val", RUN.vocab_size, DATA_DIR),
]
DATA_DIR.mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    in_bucket = !gcloud storage ls {DATA_BUCKET}/ 2>/dev/null
    for path in token_files:
        if not path.exists() and f"{DATA_BUCKET}/{path.name}" in in_bucket:
            !gcloud storage cp {DATA_BUCKET}/{path.name} {path}

tokenize_webtext.prepare(RUN.vocab_size, RUN.num_train_shards, DATA_DIR, RUN.num_tokenizer_shards)

if IN_COLAB:
    for path in token_files:
        if f"{DATA_BUCKET}/{path.name}" not in in_bucket:
            !gcloud storage cp {path} {DATA_BUCKET}/{path.name}

train_tokens = token_files[1].stat().st_size // 2
print(f"{train_tokens:,} training tokens, {token_files[2].stat().st_size // 2:,} validation tokens")

if IN_COLAB and not SMOKE and jax.default_backend() == "cpu":
    raise RuntimeError("Token files are in the bucket. Training needs an accelerator; see COLAB.md.")

In [ ]:
def make_experiment(run_id=RUN_ID, logger_config=None, checkpoint_dir=None):
    """One run through the harness. Defaults to the full run, on W&B and GCS."""
    output_dir = ROOT / "climbmix_logs"
    if IN_COLAB and logger_config is None:
        logger_config = experiment.WandBLoggerConfig(
            project_name=WANDB_PROJECT,
            user_name=WANDB_ENTITY,
            log_dir=str(output_dir / run_id),
            run_id=run_id,  # A repeat of this run resumes the same W&B run.
            train_log_interval_steps=10,
            also_log_to_file=True,
        )
    return train.create_run_experiment(
        RUN,
        run_id=run_id,
        data_dir=DATA_DIR,
        output_dir=output_dir,
        checkpoint_dir=checkpoint_dir or f"{CHECKPOINT_ROOT}/{run_id}",
        logger_config=logger_config,
    )

## How long will this take?

A short timing run, saved nowhere. It also proves that the model fits and that training
and evaluation work on this accelerator before hours go into them. Peak memory is printed
where the accelerator reports it. An out-of-memory error here is the cue to change the
batch settings.

In [ ]:
import tempfile
import time

with tempfile.TemporaryDirectory() as tmp:
    tmp = pathlib.Path(tmp)
    exp = make_experiment(
        "timing",
        logger_config=experiment.LoggerConfig(log_dir=str(tmp), run_id="timing"),
        checkpoint_dir=str(tmp / "checkpoints"),
    )
    batches = iter(exp.train_task.dataset)
    exp.inner_loop(next(batches))  # Compiles the training step.
    jax.block_until_ready(exp.model)

    start = time.monotonic()
    for _ in range(TIMING_STEPS):
        exp.inner_loop(next(batches))
    jax.block_until_ready(exp.model)
    seconds_per_step = (time.monotonic() - start) / TIMING_STEPS

    exp.run_validation()  # Compiles the evaluation step.
    start = time.monotonic()
    exp.run_validation()
    seconds_per_eval = time.monotonic() - start

    n_params = analyze.print_param_sizes(exp.model, depth=1)[1]
    peak = (jax.local_devices()[0].memory_stats() or {}).get("peak_bytes_in_use")
    exp.cleanup()

num_evals = RUN.max_steps // RUN.eval_interval_steps
hours = (RUN.max_steps * seconds_per_step + num_evals * seconds_per_eval) / 3600
tokens = RUN.max_steps * RUN.batch_size * RUN.seq_len
print(f"{n_params:,} params  {seconds_per_step:.2f} s/step  {seconds_per_eval:.1f} s/eval  "
      f"about {hours:.1f} h for {RUN.max_steps:,} steps, plus checkpoints")
print(f"{tokens:,} tokens, {tokens / n_params:.1f} per parameter, "
      f"{tokens / train_tokens:.2f} passes over the data")
if peak:
    print(f"peak memory {peak / 2**30:.1f} GiB")

## Train

Prints a W&B link, where the curves and samples appear as it trains. It starts from the
newest checkpoint in the bucket if there is one.

In [ ]:
exp = make_experiment()
try:
    resumed = exp.resume()
    print(("resuming at step " + str(exp.step)) if resumed else "starting fresh", f"of {RUN.max_steps:,}")
    exp.outer_loop()
    final = exp.run_validation()
    exp.sampling_evaluation()
finally:
    exp.cleanup()  # Waits for W&B to finish syncing.
print(final)